In [ ]:
!pip install chromadb sentence-transformers


In [ ]:

!pip install chromadb sentence-transformers

from google.colab import files

uploaded = files.upload()
doc_name = list(uploaded.keys())[0]
with open(doc_name, "r") as f:
    text = f.read()

print("Preview of document:\n")
print(text[:500])

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from bert_score import score


sentences = text.split(". ")
chunks = [". ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]

print("Number of chunks:", len(chunks))
print("First chunk:\n", chunks[0])

model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()

if "docs" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("docs")

collection = chroma_client.create_collection(name="docs")

for i, chunk in enumerate(chunks):
    emb = model.encode(chunk).tolist()
    collection.add(documents=[chunk], embeddings=[emb], ids=[str(i)])

print("Stored", len(chunks), "chunks in ChromaDB")

query = "What is retrieval augmented generation?"
query_emb = model.encode(query).tolist()

results = collection.query(query_embeddings=[query_emb], n_results=3)
retrieved_chunks = results["documents"][0]

print("Query:", query)
print("\nTop Retrieved Chunks:\n")
for doc in retrieved_chunks:
    print("-", doc, "\n")

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from bert_score import score


sentences = text.split(". ")
chunks = [". ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]

print("Number of chunks:", len(chunks))
print("First chunk:\n", chunks[0])

model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()

if "docs" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection("docs")

collection = chroma_client.create_collection(name="docs")

for i, chunk in enumerate(chunks):
    emb = model.encode(chunk).tolist()
    collection.add(documents=[chunk], embeddings=[emb], ids=[str(i)])

print("Stored", len(chunks), "chunks in ChromaDB")

query = "What is retrieval augmented generation?"
query_emb = model.encode(query).tolist()

results = collection.query(query_embeddings=[query_emb], n_results=3)
retrieved_chunks = results["documents"][0]

print("Query:", query)
print("\nTop Retrieved Chunks:\n")
for doc in retrieved_chunks:
    print("-", doc, "\n")


P, R, F1 = score(retrieved_chunks, [query]*len(retrieved_chunks), lang="en", verbose=True)

print("\nBERTScore Results:")
for i, c in enumerate(retrieved_chunks):
    print(f"Chunk {i+1}:")
    print("  Precision:", round(P[i].item(), 4))
    print("  Recall:   ", round(R[i].item(), 4))
    print("  F1:       ", round(F1[i].item(), 4))
    print("  Text:     ", c, "\n")



In [ ]:

!pip install sentence-transformers weaviate-client --quiet
!pip install bert-score --quiet



In [ ]:
from google.colab import files
import weaviate
from weaviate.classes.config import Property, DataType
from sentence_transformers import SentenceTransformer
from bert_score import score

uploaded = files.upload()
filename = list(uploaded.keys())[0]
with open(filename, "r") as f:
    text = f.read()

sentences = text.split(". ")
chunks = [". ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]

print("Number of chunks:", len(chunks))
print("First chunk:\n", chunks[0])

try:
    client = weaviate.connect_to_embedded()
except:
    client = weaviate.connect_to_local(port=8079, grpc_port=50050)

if client.collections.exists("DocumentChunk"):
    client.collections.delete("DocumentChunk")

collection = client.collections.create(
    name="DocumentChunk",
    properties=[Property(name="content", data_type=DataType.TEXT)]
)

model = SentenceTransformer("all-MiniLM-L6-v2")

with collection.batch.dynamic() as batch:
    for chunk in chunks:
        emb = model.encode(chunk).tolist()
        batch.add_object(properties={"content": chunk}, vector=emb)

print("Stored", len(chunks), "chunks in Weaviate")

query = "What is retrieval augmented generation?"
query_emb = model.encode(query).tolist()

response = collection.query.near_vector(query_emb, limit=3)
retrieved_chunks = [res.properties["content"] for res in response.objects]

print("\nQuery:", query)
print("\nTop Retrieved Chunks:\n")
for c in retrieved_chunks:
    print("-", c, "\n")


P, R, F1 = score(retrieved_chunks, [query]*len(retrieved_chunks), lang="en", verbose=True)

print("\nBERTScore Results:")
for i, c in enumerate(retrieved_chunks):
    print(f"Chunk {i+1}:")
    print("  Precision:", round(P[i].item(), 4))
    print("  Recall:   ", round(R[i].item(), 4))
    print("  F1:       ", round(F1[i].item(), 4))
    print("  Text:     ", c, "\n")
